# DAgger adapter only: train from the rows you already have

The collection half of `train_tracks_medium_dagger.ipynb` costs 70–85 min and does not
need repeating — run-06's rows were already built and verified:

| | Train | Dev |
| --- | --- | --- |
| States | 283 | 41 |
| Rows (both menu orders) | **566** | 41 |
| Puzzles | 32 | 7 |
| Labels wrong | 0 | 0 |

This notebook trains that set from the run-04 adapter and zips the result. It does not
collect, probe or evaluate anything.

**Expected wall time:** ~35–45 min (smoke ~6 min, training ~25–35 min). Run All.

**Inputs** (two datasets):

1. `run-04-branch-search-depth` — the 7 adapter files, flat;
2. a dataset holding `train-examples.jsonl` and `dev-examples.jsonl` — rename
   `train-examples.txt`/`dev-examples.txt` from run-06 to `.jsonl`.

**What to expect from the artifact:** this is a 53-step fine-tune on 42 corrected
branch decisions. It is a record of the DAgger round, not a better checkpoint —
run-04 at 28/32 remains the policy to keep. If the rows are missing or malformed the
checks below stop before the GPU is touched.


## 1. Environment

Two `Tesla T4` -> fp16, same pins as run-04/05.


In [ ]:
import subprocess, sys, torch

print(sys.version)
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
if torch.cuda.is_available():
    major = torch.cuda.get_device_capability(0)[0]
    print("selected dtype:", "bfloat16" if major >= 8 else "float16 (T4 is sm_75)")
print()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


## 2. Dependencies

`trl` is needed for training; `torchao` is removed because Kaggle's 0.10 build makes
PEFT refuse to inject the adapter.


In [ ]:
%pip install -q "transformers==5.22.2" "trl>=1.15.0,<2" "peft>=0.21.2,<1" "datasets>=4.8.4,<5" "accelerate>=1.15.0,<2" "huggingface-hub>=1.31.0,<2" "safetensors>=0.8.0,<1"
%pip uninstall -y -q torchao


In [ ]:
import importlib.metadata as metadata

for package in ("torch", "transformers", "trl", "peft", "accelerate", "datasets"):
    print(f"{package:14s}", metadata.version(package))


## 3. Locate the adapter and the two example files

Discovery is by filename, so any dataset layout works. Both files are checked for
count and shape before anything loads on the GPU.


In [ ]:
from pathlib import Path
import json

INPUT = Path("/kaggle/input")
ADAPTER_OVERRIDE = None    # e.g. Path("/kaggle/input/run-04-branch-search-depth")
EXAMPLES_OVERRIDE = None   # e.g. Path("/kaggle/input/dagger-run06")

adapter_hits = sorted({hit.parent for hit in INPUT.glob("**/adapter_config.json")})
print("adapter_config.json found in:")
for hit in adapter_hits:
    print("  ", hit, "| weights:", (hit / "adapter_model.safetensors").exists())
assert adapter_hits, (
    "No adapter_config.json under /kaggle/input. Upload the 7 run-04 adapter files "
    "as a dataset (not the zip) and attach it.")
preferred = [hit for hit in adapter_hits if "run-04" in hit.name.lower()]
ADAPTER = (Path(ADAPTER_OVERRIDE) if ADAPTER_OVERRIDE
           else (preferred[0] if preferred else adapter_hits[0]))
assert (ADAPTER / "adapter_model.safetensors").exists(), (
    f"{ADAPTER} has a config but no weights")
config = json.loads((ADAPTER / "adapter_config.json").read_text())
print("\nadapter:", ADAPTER)
print("  base model:", config.get("base_model_name_or_path"),
      "| r/alpha:", config.get("r"), "/", config.get("lora_alpha"))
assert config.get("r") == 16, "expected the run-04 adapter (r=16)"

if EXAMPLES_OVERRIDE:
    TRAIN_EX = Path(EXAMPLES_OVERRIDE) / "train-examples.jsonl"
    DEV_EX = Path(EXAMPLES_OVERRIDE) / "dev-examples.jsonl"
else:
    train_hits = sorted(INPUT.glob("**/train-examples.jsonl"))
    dev_hits = sorted(INPUT.glob("**/dev-examples.jsonl"))
    assert train_hits, "no train-examples.jsonl under /kaggle/input"
    assert dev_hits, "no dev-examples.jsonl under /kaggle/input"
    TRAIN_EX, DEV_EX = train_hits[0], dev_hits[0]

train_rows = [json.loads(line) for line in TRAIN_EX.open(encoding="utf-8")]
dev_rows = [json.loads(line) for line in DEV_EX.open(encoding="utf-8")]
print("\ntrain examples:", TRAIN_EX, "|", len(train_rows), "rows")
print("dev examples  :", DEV_EX, "|", len(dev_rows), "rows")
assert len(train_rows) >= 400, f"only {len(train_rows)} train rows; wrong file?"
assert len(dev_rows) >= 20, f"only {len(dev_rows)} dev rows; wrong file?"
bad = [row for row in train_rows[:200]
       if sorted(row) != sorted({"completion", "kind", "menu_size", "puzzle_id",
                                 "target", "user"})]
assert not bad, f"unexpected row keys: {sorted(bad[0])}"
print("train puzzles:", len({row["puzzle_id"] for row in train_rows}),
      "| kinds:", sorted({row["kind"] for row in train_rows}))
print(f"estimated tokens: {sum(len(r['user']) + len(r['completion']) for r in train_rows) / 3.6 / 1e3:.0f}k")

MODEL = config.get("base_model_name_or_path") or "Qwen/Qwen2.5-1.5B-Instruct"
print("base model:", MODEL)


## 4. Write the three modules training needs

`tt_train.py` imports only `tt_core` (rules, prompt, validator) and `tt_lib` (data
builder); the evaluator and collector are not needed here.


In [ ]:
from pathlib import Path

TT = Path("/kaggle/working/tt")
TT.mkdir(parents=True, exist_ok=True)
print(TT)


In [ ]:
%%writefile /kaggle/working/tt/tt_core.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Small, inspectable constraint engine for offline Train Tracks traces.

Rows and columns are zero-based internally. Masks use the tile order in TILES.
Every deduction removes only candidates ruled out by a visible local constraint.
"""

from __future__ import annotations

from dataclasses import dataclass

TILES = ("XX", "NS", "EW", "NE", "NW", "SE", "SW")
BIT = {name: 1 << index for index, name in enumerate(TILES)}
ALL = (1 << len(TILES)) - 1
TRACK = ALL ^ BIT["XX"]
DIRECTIONS = {"N": (-1, 0, "S"), "E": (0, 1, "W"),
              "S": (1, 0, "N"), "W": (0, -1, "E")}


def choices(mask: int) -> list[str]:
    return [name for name in TILES if mask & BIT[name]]


def one(mask: int) -> str | None:
    return TILES[mask.bit_length() - 1] if mask and mask.bit_count() == 1 else None


def clue_from_record(record: dict) -> dict:
    puzzle = record["puzzle"]
    height = len(puzzle["rows"])
    givens = [(height - 1 - row, column, shape)
              for shape, row, column, *_ in puzzle["pieces"]]
    return {"rows": puzzle["rows"], "columns": puzzle["columns"],
            "given_shapes": sorted(givens)}


def initial_domains(clue: dict) -> list[list[int]]:
    height, width = len(clue["rows"]), len(clue["columns"])
    given = {(r, c): shape for r, c, shape in clue["given_shapes"]}
    board = []
    for r in range(height):
        line = []
        for c in range(width):
            if (r, c) in given:
                line.append(BIT[given[r, c]])
                continue
            allowed = 0
            for name in TILES:
                if all(0 <= r + DIRECTIONS[d][0] < height
                       and 0 <= c + DIRECTIONS[d][1] < width for d in name if d != "X"):
                    allowed |= BIT[name]
            line.append(allowed)
        board.append(line)
    return board


def snapshot(board: list[list[int]]) -> list[list[int]]:
    return [line.copy() for line in board]


@dataclass(frozen=True)
class Step:
    kind: str  # deduce or contradiction
    rule: str
    anchor: dict
    cell: tuple[int, int] | None = None
    mask: int | None = None

    def action(self) -> dict:
        result = {"type": self.kind, "rule": self.rule, "anchor": self.anchor}
        if self.cell is not None:
            result["cell"] = {"row": self.cell[0] + 1,
                              "column": self.cell[1] + 1}
        if self.mask is not None:
            result["candidates"] = choices(self.mask)
        return result


def _line_step(board, cells, clue, kind, index):
    fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
    possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
    anchor = {kind: index + 1, "clue": clue}
    if fixed > clue or possible < clue:
        return Step("contradiction", "clue_bounds", anchor)
    for r, c in cells:
        old = board[r][c]
        if fixed == clue and old & BIT["XX"] and old & TRACK:
            return Step("deduce", "clue_satisfied", anchor,
                        (r, c), BIT["XX"])
        if possible == clue and old & BIT["XX"] and old & TRACK:
            return Step("deduce", "all_remaining_cells_required", anchor,
                        (r, c), old & TRACK)
    return None


def _port_step(board, r, c):
    height, width = len(board), len(board[0])
    old = board[r][c]
    supported = 0
    for shape in choices(old):
        if shape == "XX":
            ports = ""
        else:
            ports = shape
        valid = True
        for direction, (dr, dc, opposite) in DIRECTIONS.items():
            nr, nc = r + dr, c + dc
            if not (0 <= nr < height and 0 <= nc < width):
                continue  # The boundary and endpoint rule is set initially.
            if not any((direction in ports) == (opposite in neighbor)
                       for neighbor in choices(board[nr][nc])):
                valid = False
                break
        if valid:
            supported |= BIT[shape]
    if supported == old:
        return None
    anchor = {"row": r + 1, "column": c + 1}
    return (Step("deduce", "port_reciprocity", anchor, (r, c), supported)
            if supported else Step("contradiction", "port_reciprocity", anchor,
                                   (r, c)))


def next_step(board: list[list[int]], clue: dict) -> Step | None:
    height, width = len(board), len(board[0])
    for r in range(height):
        event = _line_step(board, [(r, c) for c in range(width)],
                           clue["rows"][r], "row", r)
        if event:
            return event
    for c in range(width):
        event = _line_step(board, [(r, c) for r in range(height)],
                           clue["columns"][c], "column", c)
        if event:
            return event
    for r in range(height):
        for c in range(width):
            event = _port_step(board, r, c)
            if event:
                return event
    if all(one(mask) is not None for line in board for mask in line):
        error = completed_error(board, clue)
        if error:
            return Step("contradiction", error, {"board": "complete"})
    return None


def completed_error(board: list[list[int]], clue: dict) -> str | None:
    height, width = len(board), len(board[0])
    grid = [[one(mask) for mask in line] for line in board]
    if any(tile is None for line in grid for tile in line):
        return "incomplete"
    if any(sum(tile != "XX" for tile in grid[r]) != clue["rows"][r]
           for r in range(height)) or any(
               sum(grid[r][c] != "XX" for r in range(height)) != clue["columns"][c]
               for c in range(width)):
        return "clue_mismatch"
    occupied = {(r, c) for r in range(height) for c in range(width)
                if grid[r][c] != "XX"}
    given = {(r, c): shape for r, c, shape in clue["given_shapes"]}
    if any(grid[r][c] != shape for (r, c), shape in given.items()):
        return "changed_given"
    external = []
    for r, c in occupied:
        for direction in grid[r][c]:
            dr, dc, opposite = DIRECTIONS[direction]
            nr, nc = r + dr, c + dc
            if not (0 <= nr < height and 0 <= nc < width):
                external.append((r, c, direction))
            elif opposite not in grid[nr][nc]:
                return "port_reciprocity"
    expected_external = {(r, c, direction) for (r, c), shape in given.items()
                         for direction in shape if not (
                             0 <= r + DIRECTIONS[direction][0] < height and
                             0 <= c + DIRECTIONS[direction][1] < width)}
    if set(external) != expected_external or len(external) != 2:
        return "endpoints"
    if not occupied:
        return "empty_path"
    found, todo = set(), [next(iter(occupied))]
    while todo:
        cell = todo.pop()
        if cell in found:
            continue
        found.add(cell)
        r, c = cell
        for direction in grid[r][c]:
            dr, dc, _ = DIRECTIONS[direction]
            neighbor = r + dr, c + dc
            if neighbor in occupied and neighbor not in found:
                todo.append(neighbor)
    return None if found == occupied else "disconnected_path"


def apply_deduction(board, step: Step) -> None:
    if step.kind != "deduce" or step.cell is None or step.mask is None:
        raise ValueError("Expected a deduction")
    r, c = step.cell
    if step.mask == 0 or step.mask == board[r][c] or step.mask & board[r][c] != step.mask:
        raise ValueError("Deduction must remove at least one candidate")
    board[r][c] = step.mask


def branch_cell(board: list[list[int]]) -> tuple[int, int] | None:
    candidates = [(mask.bit_count(), r, c) for r, line in enumerate(board)
                  for c, mask in enumerate(line) if mask.bit_count() > 1]
    return min(candidates)[1:] if candidates else None


"""Shared text policy format and rule checker for interactive trace rollouts."""

import hashlib
import json

SYSTEM = ("Solve Train Tracks one action at a time. Choose only from the "
          "provided legal-action menu. Return exactly one JSON object with the "
          "selected id, for example {\"choice\":\"a00\"}. Do not invent an "
          "action or alter a listed action.")


def development_ids(puzzle_ids, count=32):
    ids = sorted(set(puzzle_ids), key=lambda x: hashlib.sha256(x.encode()).hexdigest())
    if len(ids) <= count:
        raise ValueError("Not enough puzzles for a disjoint train/development split")
    return set(ids[:count])


def _action_key(action):
    return json.dumps(action, sort_keys=True, separators=(",", ":"))


def legal_actions(state, pending=False):
    """Enumerate rule-supported actions; offer branches only when propagation stalls."""
    board = state["domains"]
    clue = {"rows": state["row_clues"], "columns": state["column_clues"],
            "given_shapes": [(g["row"] - 1, g["column"] - 1, g["value"])
                             for g in state["givens"]]}
    if pending:
        stack = state["branch_stack"]
        return ([{"type": "undo", "to_step": stack[-1]["to_step"]}]
                if stack else [])

    height, width = len(board), len(board[0])
    stack = state["branch_stack"]
    actions = []
    for row in range(height):
        cells = [(row, c) for c in range(width)]
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        number = clue["rows"][row]
        anchor = {"row": row + 1, "clue": number}
        if stack and (fixed > number or possible < number):
            actions.append(Step("contradiction", "clue_bounds", anchor).action())
        for c in range(width):
            cell = (row, c)
            for rule in ("clue_satisfied", "all_remaining_cells_required"):
                step = _rule_step(board, clue, rule, anchor, cell)
                if step:
                    actions.append(step.action())
    for column in range(width):
        cells = [(r, column) for r in range(height)]
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        number = clue["columns"][column]
        anchor = {"column": column + 1, "clue": number}
        if stack and (fixed > number or possible < number):
            actions.append(Step("contradiction", "clue_bounds", anchor).action())
        for r in range(height):
            cell = (r, column)
            for rule in ("clue_satisfied", "all_remaining_cells_required"):
                step = _rule_step(board, clue, rule, anchor, cell)
                if step:
                    actions.append(step.action())
    for r in range(height):
        for c in range(width):
            step = _port_step(board, r, c)
            if step and (step.kind == "deduce" or stack):
                actions.append(step.action())

    error = completed_error(board, clue)
    if error is None:
        actions.append({"type": "finish"})
    elif stack and error != "incomplete":
        actions.append(Step("contradiction", error, {"board": "complete"}).action())

    # A branch is a fallback after no visible deduction remains.
    if not actions:
        cell = branch_cell(board)
        if cell:
            r, c = cell
            for value in choices(board[r][c]):
                actions.append({"type": "branch",
                                "cell": {"row": r + 1, "column": c + 1},
                                "value": value})

    unique = {_action_key(action): action for action in actions}
    return [unique[key] for key in sorted(unique)]


def choice_id(action, actions):
    key = _action_key(action)
    for index, candidate in enumerate(actions):
        if _action_key(candidate) == key:
            return f"a{index:03d}"
    raise ValueError(f"Trace action is absent from its legal menu: {action}")


def prompt(state, actions):
    readable = dict(state)
    readable["domains"] = [[choices(mask) for mask in row]
                           for row in state["domains"]]
    readable["legal_actions"] = [
        {"id": f"a{index:03d}", "action": action}
        for index, action in enumerate(actions)]
    return ("Board state (each domains cell lists its remaining tiles; "
            "XX means empty):\n"
            + json.dumps(readable, separators=(",", ":"), ensure_ascii=False)
            + "\nChoose one listed legal action. JSON choice id only.")


def completion(action_id):
    return json.dumps({"choice": action_id}, separators=(",", ":"))


def parse_action(raw, actions):
    try:
        result = json.loads(raw.strip())
    except (ValueError, TypeError):
        return None
    if not isinstance(result, dict) or set(result) != {"choice"}:
        return None
    choice = result["choice"]
    if not isinstance(choice, str) or not choice.startswith("a"):
        return None
    try:
        index = int(choice[1:])
    except ValueError:
        return None
    return actions[index] if 0 <= index < len(actions) and choice == f"a{index:03d}" else None


def _cell(action, board):
    cell = action.get("cell")
    if not isinstance(cell, dict):
        return None
    r, c = cell.get("row"), cell.get("column")
    if type(r) is not int or type(c) is not int or not (1 <= r <= len(board)) or not (1 <= c <= len(board[0])):
        return None
    return r - 1, c - 1


def _rule_step(board, clue, rule, anchor, cell=None):
    if not isinstance(rule, str):
        return None
    h, w = len(board), len(board[0])
    if rule in {"clue_satisfied", "all_remaining_cells_required", "clue_bounds"}:
        if not isinstance(anchor, dict):
            return None
        if "row" in anchor and type(anchor["row"]) is int and 1 <= anchor["row"] <= h:
            i = anchor["row"] - 1
            cells = [(i, c) for c in range(w)]
            kind, number = "row", clue["rows"][i]
        elif "column" in anchor and type(anchor["column"]) is int and 1 <= anchor["column"] <= w:
            i = anchor["column"] - 1
            cells = [(r, i) for r in range(h)]
            kind, number = "column", clue["columns"][i]
        else:
            return None
        if anchor != {kind: i + 1, "clue": number}:
            return None
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        if rule == "clue_bounds":
            return Step("contradiction", rule, anchor) if fixed > number or possible < number else None
        if cell not in cells:
            return None
        r, c = cell
        old = board[r][c]
        if not (old & BIT["XX"] and old & TRACK):
            return None
        if rule == "clue_satisfied" and fixed == number:
            return Step("deduce", rule, anchor, cell, BIT["XX"])
        if rule == "all_remaining_cells_required" and possible == number:
            return Step("deduce", rule, anchor, cell, old & TRACK)
    elif rule == "port_reciprocity" and isinstance(anchor, dict):
        if type(anchor.get("row")) is not int or type(anchor.get("column")) is not int:
            return None
        r, c = anchor["row"] - 1, anchor["column"] - 1
        if not (0 <= r < h and 0 <= c < w):
            return None
        if cell is not None and cell != (r, c):
            return None
        return _port_step(board, r, c)
    elif rule in {"disconnected_path", "endpoints", "empty_path", "clue_mismatch", "changed_given"}:
        if anchor == {"board": "complete"} and completed_error(board, clue) == rule:
            return Step("contradiction", rule, anchor)
    return None


def validate_action(board, clue, action, stack, pending):
    """Return (valid, reason, canonical action). No solution is consulted."""
    if not isinstance(action, dict):
        return False, "malformed", None
    kind = action.get("type")
    if not isinstance(kind, str):
        return False, "malformed", None
    if pending:
        if kind != "undo" or not stack or action.get("to_step") != stack[-1]["to_step"]:
            return False, "expected_undo", None
        return True, "ok", {"type": "undo", "to_step": stack[-1]["to_step"]}
    if kind in {"deduce", "contradiction"}:
        cell = _cell(action, board) if "cell" in action else None
        if "cell" in action and cell is None:
            return False, "invalid_cell", None
        step = _rule_step(board, clue, action.get("rule"), action.get("anchor"), cell)
        if step is None or step.kind != kind:
            return False, "unsupported_rule", None
        if kind == "deduce":
            if cell is None or step.cell != cell or action.get("candidates") != choices(step.mask):
                return False, "wrong_candidates", None
        elif step.cell is not None and step.cell != cell:
            return False, "wrong_contradiction_cell", None
        elif step.cell is None and cell is not None:
            return False, "unexpected_cell", None
        return True, "ok", step.action()
    if kind == "branch":
        cell = _cell(action, board)
        value = action.get("value")
        if cell is None or not isinstance(value, str) or value not in BIT or board[cell[0]][cell[1]].bit_count() < 2:
            return False, "invalid_branch", None
        if not board[cell[0]][cell[1]] & BIT[value]:
            return False, "branch_outside_domain", None
        if completed_error(board, clue) not in {"incomplete", None}:
            return False, "contradictory_board", None
        return True, "ok", {"type": "branch", "cell": action["cell"], "value": value}
    if kind == "finish":
        return (True, "ok", {"type": "finish"}) if completed_error(board, clue) is None else (False, "unfinished", None)
    return False, "unexpected_action", None


def split_examples(rows, max_deductions_per_puzzle=48):
    """Keep every start and rare positive action; balance sampled deductions by rule."""
    from collections import defaultdict
    grouped = defaultdict(list)
    for row in rows:
        if row.get("expert_policy_label"):
            grouped[row["puzzle_id"]].append(row)
    selected = []
    for puzzle_id, items in grouped.items():
        deductions = [r for r in items if r["action"]["type"] == "deduce"]
        rare = [r for r in items if r["action"]["type"] != "deduce"]
        starts = [r for r in items if r["step_index"] == 0]
        selected.extend(starts)

        start_steps = {r["step_index"] for r in starts}
        by_rule = defaultdict(list)
        for row in deductions:
            if row["step_index"] not in start_steps:
                by_rule[row["action"].get("rule", "unknown")].append(row)
        for rule, rule_rows in by_rule.items():
            rule_rows.sort(key=lambda r: hashlib.sha256(
                f"{puzzle_id}:{r['step_index']}:42".encode()).hexdigest())

        start_deductions = sum(r["action"]["type"] == "deduce" for r in starts)
        remaining = max(0, max_deductions_per_puzzle - start_deductions)
        rules = sorted(by_rule)
        while remaining and rules:
            active = []
            for rule in rules:
                if by_rule[rule]:
                    selected.append(by_rule[rule].pop())
                    remaining -= 1
                    if by_rule[rule]:
                        active.append(rule)
                    if remaining == 0:
                        break
            rules = active
        selected.extend(r for r in rare if r["step_index"] not in start_steps)
    selected.sort(key=lambda r: hashlib.sha256(
        f"{r['puzzle_id']}:{r['step_index']}:shuffle".encode()).hexdigest())
    return selected


In [ ]:
%%writefile /kaggle/working/tt/tt_lib.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Safe bounded search for Train Tracks branch values.

The evaluator offers a branch only when no deduction remains, and only two
values of one cell.  A wrong value can cost dozens of forced deductions before
a contradiction, so we spend a little CPU here to reject values that are
provably impossible under the same rule engine.

The filter is *safe*: it only rejects a value when propagation plus a bounded
search reaches a contradiction.  If the current board still contains the true
solution, the true value cannot be rejected.  When both values survive the
search, callers should fall back to the model's choice.

Validated on the run-03 medium rollouts: on boards that still contain the
reference solution, depth 2 raised branch accuracy from 68.2% to 92.4% and never
rejected the true value.  Depth 3 added nothing, so depth 2 is the default.
"""

from __future__ import annotations

from tt_core import legal_actions
from tt_core import BIT, TILES, branch_cell, choices, completed_error

DEFAULT_DEPTH = 2


def clue_from_state(state: dict) -> dict:
    return {
        "rows": state["row_clues"],
        "columns": state["column_clues"],
        "given_shapes": [
            (g["row"] - 1, g["column"] - 1, g["value"]) for g in state["givens"]
        ],
    }


def _mask(candidates) -> int:
    return sum(BIT[c] for c in candidates)


def propagate(board, state: dict, stack: list, round_limit: int = 400):
    """Apply every currently legal deduction until a fixpoint or contradiction.

    Returns ``(contradicted, board, deductions)``.  Actions that became stale
    after an earlier action in the same round are skipped rather than treated as
    contradictions.
    """
    board = [row[:] for row in board]
    deductions = 0
    for _ in range(round_limit):
        local = {
            "row_clues": state["row_clues"],
            "column_clues": state["column_clues"],
            "givens": state["givens"],
            "domains": board,
            "branch_stack": stack,
        }
        actions = legal_actions(local, pending=False)
        if any(a["type"] == "contradiction" for a in actions):
            return True, board, deductions
        deduces = [a for a in actions if a["type"] == "deduce"]
        if not deduces:
            return False, board, deductions
        applied = 0
        for action in deduces:
            r = action["cell"]["row"] - 1
            c = action["cell"]["column"] - 1
            new = _mask(action["candidates"])
            if new == board[r][c] or new & board[r][c] != new:
                continue
            board[r][c] = new
            deductions += 1
            applied += 1
        if not applied:
            return False, board, deductions
    return False, board, deductions


def possible(board, state: dict, stack: list, depth: int, node_budget: list | None = None) -> bool:
    """Return True if the board is not proven impossible within ``depth`` guesses."""
    if node_budget is not None:
        if node_budget[0] <= 0:
            return True
        node_budget[0] -= 1
    contradicted, board, _ = propagate(board, state, stack)
    if contradicted:
        return False
    clue = clue_from_state(state)
    if branch_cell(board) is None:
        return completed_error(board, clue) is None
    if depth <= 0:
        return True  # unknown: conservatively possible
    r, c = branch_cell(board)
    cell = {"row": r + 1, "column": c + 1}
    for value in choices(board[r][c]):
        child = [row[:] for row in board]
        child[r][c] = BIT[value]
        child_stack = list(stack) + [
            {"to_step": -1, "cell": cell, "value": value}
        ]
        if possible(child, state, child_stack, depth - 1, node_budget):
            return True
    return False


def surviving_values(state: dict, cell: dict, depth: int = DEFAULT_DEPTH,
                     node_budget: int = 4000) -> dict:
    """Probe both values at ``cell`` and return details about survivors."""
    r, c = cell["row"] - 1, cell["column"] - 1
    candidates = [t for t in TILES if state["domains"][r][c] & BIT[t]]
    budget = [node_budget]
    feasible = {}
    for value in candidates:
        child = [row[:] for row in state["domains"]]
        child[r][c] = BIT[value]
        child_stack = list(state["branch_stack"]) + [
            {"to_step": -1, "cell": cell, "value": value}
        ]
        feasible[value] = possible(child, state, child_stack, depth, budget)
    survivors = [v for v in candidates if feasible[v]]
    return {"candidates": candidates, "feasible": feasible, "survivors": survivors,
            "depth": depth, "nodes_left": budget[0]}


def choose_branch(state: dict, cell: dict, model_value: str,
                  depth: int = DEFAULT_DEPTH, node_budget: int = 4000) -> dict:
    """Pick a branch value, using the model choice when it survives.

    Returns a dict with the chosen ``value``, whether the search overrode the
    model, and the survivor details for logging.
    """
    if depth <= 0:
        return {"value": model_value, "override": False, "search": None}
    info = surviving_values(state, cell, depth=depth, node_budget=node_budget)
    survivors = info["survivors"]
    if len(survivors) == 1:
        value = survivors[0]
    elif model_value in survivors:
        value = model_value
    elif survivors:
        value = survivors[0]
    else:
        value = model_value  # poisoned board: the filter cannot help
    return {"value": value, "override": value != model_value, "search": info}


"""Build the Kaggle training views, including branch-focused augmentation.

Two changes relative to the original Modal baseline:

1. **Every branch decision is trained in both menu orders.**  The evaluator
   sorts the legal menu, so a model can learn "pick a001" instead of reading the
   board.  Reversing the two branch entries removes that shortcut.
2. **Wrong-branch trace rows are reused as corrected positives.**  The original
   `split_examples` drops rows with ``expert_policy_label: false``.  Their state
   is still a valid pre-branch state, and the audited solution tells us the
   correct value.  Adding that corrected example gives the model the hard states
   where the teacher probe guessed wrong, without ever training a wrong target.

Branch examples are optionally repeated so they are not drowned by the ~8.6k
deduction examples.
"""

import argparse
import json
import random
from collections import Counter
from pathlib import Path

from tt_core import (choice_id, completion, development_ids,
                           legal_actions, prompt, split_examples)

HERE = Path(__file__).resolve().parent
TILES = ("XX", "NS", "EW", "NE", "NW", "SE", "SW")


def load_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


def load_solutions(path: Path) -> dict[str, list[list[str]]]:
    solutions = {}
    for record in load_jsonl(path):
        solutions[record["puzzle_id"]] = [
            [tile or "XX" for tile in row] for row in record["solution"]
        ]
    return solutions


def _make_example(row, actions, target):
    return {
        "user": prompt(row["state"], actions),
        "completion": completion(target),
        "kind": row["action"]["type"],
        "puzzle_id": row["puzzle_id"],
        "target": target,
        "menu_size": len(actions),
    }


def _branch_pair(mask: int) -> str:
    return "|".join(name for index, name in enumerate(TILES) if mask & (1 << index))


def _corrected_branch_row(row, solution):
    cell = row["action"]["cell"]
    correct = solution[cell["row"] - 1][cell["column"] - 1]
    actions = legal_actions(row["state"], pending=False)
    branch_actions = [a for a in actions if a["type"] == "branch"]
    if not any(a["cell"] == cell and a["value"] == correct for a in branch_actions):
        return None, None
    corrected = {"type": "branch", "cell": cell, "value": correct}
    corrected_row = dict(row)
    corrected_row["action"] = corrected
    return corrected_row, actions


def build_examples(rows, solutions=None, *, branch_repeat: int = 2,
                   reverse_branch_orders: bool = True,
                   corrected_wrong_branches: bool = True,
                   max_deductions_per_puzzle: int = 48,
                   seed: int = 42):
    """Return ``(examples, stats)`` for a list of trace rows."""
    selected = split_examples(rows, max_deductions_per_puzzle)
    examples = []
    by_kind = Counter()
    pairs = Counter()
    values = Counter()
    corrected = 0
    reversed_orders = 0

    def add(row, actions, target):
        examples.append(_make_example(row, actions, target))

    for row in selected:
        pending = row["action"]["type"] == "undo"
        actions = legal_actions(row["state"], pending=pending)
        target = choice_id(row["action"], actions)
        add(row, actions, target)
        by_kind[row["action"]["type"]] += 1
        if row["action"]["type"] == "branch":
            cell = row["action"]["cell"]
            pairs[_branch_pair(row["state"]["domains"][cell["row"] - 1][cell["column"] - 1])] += 1
            values[row["action"]["value"]] += 1
            if reverse_branch_orders:
                rev = list(reversed(actions))
                add(row, rev, choice_id(row["action"], rev))
                reversed_orders += 1

    if corrected_wrong_branches and solutions is not None:
        for row in rows:
            if row["action"]["type"] != "branch" or row.get("expert_policy_label", True):
                continue
            solution = solutions.get(row["puzzle_id"])
            if solution is None:
                continue
            corrected_row, actions = _corrected_branch_row(row, solution)
            if corrected_row is None:
                continue
            target = choice_id(corrected_row["action"], actions)
            add(corrected_row, actions, target)
            by_kind["branch_corrected"] += 1
            cell = corrected_row["action"]["cell"]
            pairs[_branch_pair(corrected_row["state"]["domains"][cell["row"] - 1][cell["column"] - 1])] += 1
            values[corrected_row["action"]["value"]] += 1
            corrected += 1
            if reverse_branch_orders:
                rev = list(reversed(actions))
                add(corrected_row, rev, choice_id(corrected_row["action"], rev))
                reversed_orders += 1

    if branch_repeat > 1:
        branch_examples = [
            e for e in examples if e["kind"] in {"branch", "branch_corrected"}
        ]
        for _ in range(branch_repeat - 1):
            examples.extend(branch_examples)

    rng = random.Random(seed)
    rng.shuffle(examples)
    stats = {
        "total": len(examples),
        "base_selected": len(selected),
        "by_kind": dict(by_kind),
        "corrected_wrong_branches": corrected,
        "reversed_branch_orders": reversed_orders,
        "branch_repeat": branch_repeat,
        "branch_examples_after_repeat": sum(
            1 for e in examples if e["kind"] in {"branch", "branch_corrected"}),
        "branch_pair_counts": dict(pairs),
        "branch_value_counts": dict(values),
    }
    return examples, stats


def build_dev_examples(rows, max_deductions_per_puzzle: int = 48):
    """Clean dev view: base examples only, no augmentation."""
    selected = split_examples(rows, max_deductions_per_puzzle)
    examples = []
    for row in selected:
        actions = legal_actions(row["state"], pending=row["action"]["type"] == "undo")
        target = choice_id(row["action"], actions)
        examples.append(_make_example(row, actions, target))
    return examples


def split_rows(rows):
    dev_ids = development_ids(row["puzzle_id"] for row in rows)
    train_rows = [row for row in rows if row["puzzle_id"] not in dev_ids]
    dev_rows = [row for row in rows if row["puzzle_id"] in dev_ids]
    return train_rows, dev_rows, len(dev_ids)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=HERE / "data/train.jsonl")
    parser.add_argument("--source", type=Path,
                        default=HERE / "data/train-puzzles-small.jsonl")
    parser.add_argument("--branch-repeat", type=int, default=2)
    parser.add_argument("--no-branch-augment", action="store_true")
    parser.add_argument("--write", type=Path, default=None,
                        help="Optional JSONL output with user/completion fields")
    args = parser.parse_args()

    rows = load_jsonl(args.data)
    solutions = load_solutions(args.source)
    train_rows, dev_rows, dev_count = split_rows(rows)
    if args.no_branch_augment:
        examples, stats = build_examples(
            train_rows, solutions, branch_repeat=1,
            reverse_branch_orders=False, corrected_wrong_branches=False)
    else:
        examples, stats = build_examples(
            train_rows, solutions, branch_repeat=args.branch_repeat)
    dev = build_dev_examples(dev_rows)
    stats["puzzles_total"] = len({r["puzzle_id"] for r in rows})
    stats["puzzles_train"] = len({r["puzzle_id"] for r in train_rows})
    stats["puzzles_dev"] = dev_count
    stats["dev_examples"] = len(dev)
    print(json.dumps(stats, indent=2))
    if args.write:
        with args.write.open("w", encoding="utf-8", newline="\n") as handle:
            for example in examples:
                handle.write(json.dumps(example, ensure_ascii=False) + "\n")
        print(f"wrote {len(examples)} examples to {args.write}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tt/tt_train.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Train the branch-focused Qwen2.5-1.5B action policy on Kaggle T4 x2.

This is the Kaggle port of ``modal/train.py`` with three changes:

* no Modal dependency and local data paths;
* fp16 on T4 (bf16 only when the GPU supports it);
* branch-focused data augmentation from ``data_prep.py`` (both menu orders,
  corrected wrong-branch states, branch upsampling).

Run from the ``kaggle/`` directory:

    python train_kaggle.py --output /kaggle/working/run-branch

For 2-GPU DDP:

    accelerate launch --config_file configs/accelerate_t4x2.yaml train_kaggle.py ...
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from tt_core import SYSTEM  # noqa: E402
from tt_lib import (build_dev_examples, build_examples, load_jsonl,
                       load_solutions, split_rows)  # noqa: E402


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=HERE / "data/train.jsonl")
    parser.add_argument("--source", type=Path,
                        default=HERE / "data/train-puzzles-small.jsonl")
    parser.add_argument("--examples", type=Path, default=None,
                        help="Precomputed user/completion JSONL; skips --data/--source")
    parser.add_argument("--dev-examples", type=Path, default=None,
                        help="Precomputed dev JSONL paired with --examples")
    parser.add_argument("--init-adapter", type=Path, default=None,
                        help="Continue training this LoRA adapter instead of adding a new one")
    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/run-branch"))
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--revision", default="main")
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--max-steps", type=int, default=-1)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--grad-accum", type=int, default=8)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--max-length", type=int, default=2048)
    parser.add_argument("--branch-repeat", type=int, default=2)
    parser.add_argument("--no-branch-augment", action="store_true")
    parser.add_argument("--smoke", action="store_true",
                        help="16 train / 8 dev examples, 4 optimizer steps")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--local-files-only", action="store_true")
    return parser.parse_args()


def load_model_and_tokenizer(args):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # With --init-adapter the adapter directory also holds the tokenizer that
    # rendered its training prompts, so the chat template matches exactly.
    tokenizer_source = args.model
    if args.init_adapter and (args.init_adapter / "tokenizer_config.json").exists():
        tokenizer_source = args.init_adapter
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source, revision=None if tokenizer_source != args.model else args.revision,
        local_files_only=args.local_files_only)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # bf16 requires sm_80+.  `is_bf16_supported()` can return True on some T4
    # images, where bf16 then falls back to slow emulation, so check the
    # capability explicitly.
    capability = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
    bf16_ok = capability[0] >= 8 and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16_ok else torch.float16
    kwargs = dict(revision=args.revision, local_files_only=args.local_files_only,
                  attn_implementation="sdpa")
    try:
        model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, **kwargs)
    except TypeError:  # older transformers
        model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=dtype, **kwargs)
    if args.init_adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(args.init_adapter), is_trainable=True)
        # Gradient checkpointing recomputes forward passes; some transformers/PEFT
        # combinations need the embedding output to require grad for LoRA grads.
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
    model.config.use_cache = False
    return model, tokenizer, dtype


def to_dataset(examples, tokenizer):
    from datasets import Dataset

    rows = []
    for example in examples:
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM},
             {"role": "user", "content": example["user"]}],
            tokenize=False, add_generation_prompt=True)
        rows.append({
            "prompt": prompt,
            "completion": example["completion"] + tokenizer.eos_token,
            "kind": example["kind"],
        })
    return Dataset.from_list(rows)


def sft_config(args, output_dir, fp16, bf16):
    from trl import SFTConfig

    fields = set(SFTConfig.__dataclass_fields__)
    kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": args.epochs,
        "max_steps": 4 if args.smoke else args.max_steps,
        "per_device_train_batch_size": args.batch_size,
        "gradient_accumulation_steps": args.grad_accum,
        "per_device_eval_batch_size": max(1, args.batch_size),
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "learning_rate": args.lr,
        "lr_scheduler_type": "cosine",
        "warmup_steps": 1 if args.smoke else 10,
        "logging_steps": 1 if args.smoke else 10,
        "eval_strategy": "no",
        "save_strategy": "no",
        "bf16": bf16,
        "fp16": fp16,
        "report_to": "none",
        "completion_only_loss": True,
        "packing": False,
        "max_length": args.max_length,
        "seed": args.seed,
        "data_seed": args.seed,
        "dataloader_num_workers": 2,
    }
    if "max_seq_length" in fields and "max_length" not in fields:
        kwargs["max_seq_length"] = kwargs.pop("max_length")
    if "eval_strategy" not in fields and "evaluation_strategy" in fields:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    kwargs = {key: value for key, value in kwargs.items() if key in fields}
    return SFTConfig(**kwargs)


def pin_single_gpu_if_unlaunched():
    """Keep TRL's SFTTrainer out of ``nn.DataParallel``.

    A single process that can see several GPUs makes ``Trainer`` wrap the model
    in ``nn.DataParallel``, and ``SFTTrainer`` refuses that outright.  Launchers
    (``accelerate launch``, ``torchrun``) set ``WORLD_SIZE`` and assign devices
    themselves; for a plain run we pin the process to GPU 0.

    This has to happen before the first CUDA call: ``cudaGetDeviceCount``
    caches the visible-device list, so setting the variable afterwards is too
    late.  That is why there is no device query here.
    """
    if int(os.environ.get("WORLD_SIZE", "1")) > 1:
        return
    if os.environ.get("CUDA_VISIBLE_DEVICES") is not None:
        return
    print("single-process run: pinning to GPU 0 "
          "(SFTTrainer cannot use nn.DataParallel; use accelerate launch for 2 GPUs)",
          flush=True)
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"


def main():
    args = parse_args()
    if args.branch_repeat < 1:
        raise SystemExit("--branch-repeat must be at least 1")
    pin_single_gpu_if_unlaunched()
    import torch
    from peft import LoraConfig
    from transformers import set_seed
    from trl import SFTTrainer

    set_seed(args.seed)
    args.output.mkdir(parents=True, exist_ok=True)

    if args.examples:
        train_examples = load_jsonl(args.examples)
        dev_examples = load_jsonl(args.dev_examples) if args.dev_examples else []
        train_puzzles = len({example.get("puzzle_id") for example in train_examples})
        dev_puzzles = len({example.get("puzzle_id") for example in dev_examples})
        train_rows = []
        stats = {"source": "precomputed", "total": len(train_examples),
                 "puzzles": train_puzzles}
        print(f"precomputed examples: {len(train_examples):,} train, "
              f"{len(dev_examples):,} dev")
    else:
        rows = load_jsonl(args.data)
        solutions = load_solutions(args.source)
        train_rows, dev_rows, dev_puzzles = split_rows(rows)
        if args.no_branch_augment:
            train_examples, stats = build_examples(
                train_rows, solutions, branch_repeat=1,
                reverse_branch_orders=False, corrected_wrong_branches=False)
        else:
            train_examples, stats = build_examples(
                train_rows, solutions, branch_repeat=args.branch_repeat)
        dev_examples = build_dev_examples(dev_rows)
        train_puzzles = len({row["puzzle_id"] for row in train_rows})
    if args.smoke:
        train_examples = train_examples[:16]
        dev_examples = dev_examples[:8]

    model, tokenizer, dtype = load_model_and_tokenizer(args)
    train_data = to_dataset(train_examples, tokenizer)
    dev_data = to_dataset(dev_examples, tokenizer)
    print(f"train examples: {len(train_data):,}  dev examples: {len(dev_data):,}")
    print(f"dtype: {dtype}  gpus: {torch.cuda.device_count()}  "
          f"gpu0: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

    fp16 = dtype == torch.float16
    bf16 = dtype == torch.bfloat16
    peft_config = None if args.init_adapter else LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"])
    trainer = SFTTrainer(
        model=model,
        args=sft_config(args, args.output / "trainer", fp16=fp16, bf16=bf16),
        train_dataset=train_data,
        eval_dataset=dev_data,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    started = time.monotonic()
    before = trainer.evaluate().get("eval_loss")
    trainer.train()
    after = trainer.evaluate().get("eval_loss")
    adapter = args.output / "adapter"

    # Under DDP every process would otherwise write the same files concurrently.
    if trainer.is_world_process_zero():
        trainer.save_model(str(adapter))
        tokenizer.save_pretrained(adapter)
        report = {
            "run_id": args.output.name,
            "model": args.model,
            "task": "trace_action_policy",
            "branch_augment": not args.no_branch_augment,
            "branch_repeat": 1 if args.no_branch_augment else args.branch_repeat,
            "puzzles_train": train_puzzles,
            "puzzles_dev": dev_puzzles,
            "train_examples": len(train_data),
            "dev_examples": len(dev_data),
            "data_stats": stats,
            "examples_file": str(args.examples) if args.examples else None,
            "init_adapter": str(args.init_adapter) if args.init_adapter else None,
            "continued_training": bool(args.init_adapter),
            "optimizer_steps": trainer.state.global_step,
            "development_loss_before": before,
            "development_loss_after": after,
            "train_runtime_seconds": round(time.monotonic() - started, 1),
            "peak_gpu_memory_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
            "dtype": str(dtype),
            "gpus": torch.cuda.device_count(),
            "adapter": str(adapter),
            "output_dir": str(args.output),
        }
        (args.output / "training.json").write_text(json.dumps(report, indent=2) + "\n")
        print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()


## 5. Checks

Same `run()` wrapper as the other notebooks: streams, logs, raises on non-zero exit.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(parents=True, exist_ok=True)


def run(cmd, name, extra_env=None):
    """Run a command, stream and log its output, raise on non-zero exit."""
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd), flush=True)
    log_path = LOGS / f"{name}.log"
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
    if extra_env:
        env.update(extra_env)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
    if proc.wait():
        raise RuntimeError(f"{name} failed; full output in {log_path}")
    print(f"[{name} ok] log: {log_path}", flush=True)


import torch

print("python     :", sys.executable)
print("accelerate :", shutil.which("accelerate") or "NOT FOUND")
print("torch      :", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
assert torch.cuda.is_available(), "torch cannot see a GPU; check the accelerator setting and the pip step"
assert torch.cuda.device_count() >= 1, "no CUDA devices"

LAUNCH = [sys.executable, "-m", "accelerate.commands.launch",
          "--multi_gpu", "--num_processes", "2",
          "--mixed_precision", "fp16", "--main_process_port", "29501"]
print("launcher   :", " ".join(LAUNCH[:3]), "..." if torch.cuda.device_count() >= 2
      else "(only one GPU: DDP steps will be skipped)")


In [ ]:
%cd /kaggle/working/tt
run([sys.executable, "-m", "py_compile", "tt_core.py", "tt_lib.py", "tt_train.py"], "compile")
run([sys.executable, "-c", "import importlib.util; found = importlib.util.find_spec('torchao') is not None; print('torchao installed:', found); raise SystemExit(1 if found else 0)"], "torchao-check")
run([sys.executable, "-c", "import tt_core, tt_lib, tt_train; print('module imports ok')"], "imports")
run([sys.executable, "tt_train.py", "--help"], "train-help")


### Cache the base weights once

Both DDP processes need the same ~3.1 GB checkpoint; fetching it twice at once makes
them fight over the HF cache lock.


In [ ]:
%cd /kaggle/working/tt
run([sys.executable, "-c",
     "from huggingface_hub import snapshot_download; "
     f"snapshot_download('{MODEL}'); print('cached {MODEL}')"], "model-cache")


## 6. Smoke, then train

16 rows, 4 optimizer steps first (single GPU and DDP), then 3 epochs at 2e-4.


In [ ]:
%cd /kaggle/working/tt

smoke = [sys.executable, "tt_train.py",
         "--examples", str(TRAIN_EX),
         "--dev-examples", str(DEV_EX),
         "--init-adapter", str(ADAPTER),
         "--lr", "2e-4",
         "--max-length", "4096",
         "--smoke"]
run(smoke + ["--output", "/kaggle/working/smoke-dagger"], "smoke-dagger")
if torch.cuda.device_count() >= 2:
    run(LAUNCH + smoke[1:] + ["--output", "/kaggle/working/smoke-dagger-ddp"],
        "smoke-dagger-ddp")
else:
    print("only one GPU: skipping the DDP smoke test")

train_cmd = [sys.executable, "tt_train.py",
             "--examples", str(TRAIN_EX),
             "--dev-examples", str(DEV_EX),
             "--init-adapter", str(ADAPTER),
             "--output", "/kaggle/working/run-dagger",
             "--epochs", "3",
             "--lr", "2e-4",
             "--max-length", "4096",
             "--batch-size", "1",
             "--grad-accum", "16"]
if torch.cuda.device_count() >= 2:
    try:
        run(LAUNCH + train_cmd[1:], "train-dagger-ddp")
    except (RuntimeError, FileNotFoundError) as error:
        print("\nDDP failed:", error)
        print("Falling back to a single GPU.\n")
        run(train_cmd, "train-dagger-single")
else:
    run(train_cmd, "train-dagger-single")


In [ ]:
import json

NEW_ADAPTER = Path("/kaggle/working/run-dagger/adapter")
print("adapter:", NEW_ADAPTER, "| files:", len(list(NEW_ADAPTER.glob('*'))))
report = json.loads((Path("/kaggle/working/run-dagger") / "training.json")
                    .read_text(encoding="utf-8"))
print(json.dumps({key: report[key] for key in
                  ("puzzles_train", "puzzles_dev", "train_examples", "dev_examples",
                   "optimizer_steps", "development_loss_before",
                   "development_loss_after", "train_runtime_seconds",
                   "peak_gpu_memory_gib")}, indent=1))


## 7. Save the adapter

Downloads `run-dagger-adapter.zip` plus the training report and logs.


In [ ]:
import pathlib, shutil

art = pathlib.Path("/kaggle/working/artifacts")
art.mkdir(exist_ok=True)
for pattern in ("run-dagger/training.json",):
    for path in pathlib.Path("/kaggle/working").glob(pattern):
        shutil.copy(path, art / path.name)

adapter_dir = pathlib.Path("/kaggle/working/run-dagger/adapter")
if adapter_dir.exists() and any(adapter_dir.glob("*")):
    print("adapter zip:", shutil.make_archive(str(art / "run-dagger-adapter"), "zip",
                                             str(adapter_dir)))
else:
    print("NO ADAPTER: training did not finish; re-run section 6, then this cell.")
if LOGS.exists():
    shutil.copytree(LOGS, art / "logs", dirs_exist_ok=True)
print(sorted(p.name for p in art.iterdir()))
print("download the contents of", art)
